# Spark Structured Streaming
## Batch processing
Data is collected over a period of time and then processed in a bulk at scheduled intervals.
## Stream Processing
Data is processed continuously as it arrives, enabling realtime and near realtime analytics.
## Spark Structured Streaming
Scaleable, fault tolerant stream processing engine build on top of Apache Spark.
- uses dataframe and dataset api
- micro batch processing
- exactly once guarantees
- build-in fault tolerance
- scalable to large scale workflows
- supports vaious source and sinks

## Checkpointing
Checkpointing is a fault tolerance mechanism that allow a query to recover from failures and resume processing from where it left off without data loss or dublication.
Write Ahead Logs (WAL) and Checkpointing helps to provide Fault Tolerance
Idempotent Sinks enables exactly once guarantees

### Read files useing DataStreamReader API

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType

customer_schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("customer_name", StringType()),
    StructField("date_of_birth", DateType()),
    StructField("telephone", StringType()),
    StructField("email", StringType()),
    StructField("member_since", DateType()),
    StructField("created_at", TimestampType()),
])

In [0]:
customer_df = spark \
    .readStream \
    .format("json") \
    .schema(customer_schema) \
    .load("/Volumes/gizmobox_catalog/landing/operational_data/customer_stream/")

### Transform the dataframe

In [0]:
from pyspark.sql.functions import col, current_timestamp

customer_transformed_df = customer_df \
    .withColumn("file_path", col("_metadata.file_path")) \
    .withColumn("ingestest_at", current_timestamp())

### Write data stream into Delta Table
the checkpoint location must be a unique location for every stream

#### trigger
- default -> no trigger
- fixed interval -> start at defined time intevall
- available now -> process all available data and stops

#### output mode
- append (default) -> writes new rows arrived since last micro batch
- complete -> write the entire result to the sink every time
- update -> write only the rows which have changed since last micro batch

In [0]:
streaming_query = (
    customer_transformed_df
        .writeStream
        .trigger(availableNow=True)
        .outputMode("append")
        .format("delta")
        .option("checkpointLocation" ,"/Volumes/gizmobox_catalog/landing/operational_data/customer_stream/_checkpoint")
        .toTable("gizmobox_catalog.bronze.stream_customer")
)

In [0]:
%sql
select * from gizmobox_catalog.bronze.stream_customer;

### Stop stream processing

In [0]:
streaming_query.stop()